In [50]:
import pandas as pd
import numpy as np    


In [5]:
df = pd.read_csv(r"c:\Users\sp9\OneDrive\Documentos\animal_data.csv")

In [6]:
display(df.head())

,Animal type,Country,Weight kg,Body Length cm,Gender,Animal code,Latitude,Longitude,Animal name,Observation date
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,03.01.2024
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,03.02.2024
2,European bison,Poland,930.0,335.0,male,NaN,52.828845,23.820144,Szefu,01.03.2024
3,European bison,Poland,909.0,311.0,not determined,NaN,52.830509,23.826849,NaN,01.03.2024
4,European bison™,Poland,581.0,277.0,female,NaN,52.834109,23.807093,NaN,01.03.2024


In [7]:
print (df.shape)
print (df.dtypes)

(1011, 10)
Animal type             str
Country                 str
Weight kg           float64
Body Length cm      float64
Gender                  str
Animal code         float64
Latitude            float64
Longitude           float64
Animal name             str
Observation date        str
dtype: object


In [8]:
df.info()
print(df.isnull().sum())
print(df.duplicated().sum())
print(df.drop(columns="Observation date").isna().all(axis=1).sum())

<class 'pandas.DataFrame'>
RangeIndex: 1011 entries, 0 to 1010
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Animal type       991 non-null    str    
 1   Country           999 non-null    str    
 2   Weight kg         984 non-null    float64
 3   Body Length cm    984 non-null    float64
 4   Gender            992 non-null    str    
 5   Animal code       0 non-null      float64
 6   Latitude          913 non-null    float64
 7   Longitude         913 non-null    float64
 8   Animal name       52 non-null     str    
 9   Observation date  1011 non-null   str    
dtypes: float64(5), str(5)
memory usage: 112.0 KB
Animal type           20
Country               12
Weight kg             27
Body Length cm        27
Gender                19
Animal code         1011
Latitude              98
Longitude             98
Animal name          959
Observation date       0
dtype: int64
167
12


In [7]:
for col in ["Animal type", "Country", "Gender"]:
    print(f"{col}")
    print(df[col].value_counts(dropna=False))
    print()

Animal type
Animal type
red squirrel       543
hedgehog           274
lynx                61
European bison      49
red squirrell       22
NaN                 20
red squirel         14
lynx?               10
European bison™      6
European bisson      4
European buster      4
ledgehod             3
wedgehod             1
Name: count, dtype: int64

Country
Country
Poland            291
Germany           176
Slovakia          151
Hungary           146
Czech Republic    103
Austria            74
PL                 15
NaN                12
HU                 11
Hungry             10
CZ                  5
DE                  5
Czech               4
Australia           4
CC                  4
Name: count, dtype: int64

Gender
Gender
male              499
female            488
NaN                19
not determined      5
Name: count, dtype: int64



In [12]:
print(df["Observation date"].str.replace(r"\d","9", regex=True).value_counts())
print("Fechas no estándarizadas:", df.loc[~df["Observation date"].str.match(r"^\d\d\.\d\d\.\d{4}$"), "Observation date"].unique()[:4])
print("Fecha MM.DD.AAAA (mes>12 en 2 posición):", df["Observation date"].isin(["03.13.2024", "03.14.2024", "03.22.2024"]).sum(), "filas")
print()
print("Rango de valores numéricos:")
print(df[["Weight kg", "Body Length cm", "Latitude", "Longitude"]].describe().loc[["min", "max"]])
print()
print("Peso <= 0 :", (df["Weight kg"] <= 0).sum(), "longitudes <=0:", (df["Body Length cm"] <=0).sum())
print("Cordenasdas fuera de Europa Central (lat<40 o lon>30):", ((df.Latitude < 40) | (df.Longitude > 30)).sum())


Observation date
99.99.9999    972
9 May 9999     32
9 may 9999      7
Name: count, dtype: int64
Fechas no estándarizadas: <ArrowStringArray>
['1 May 2024', '2 may 2024', '3 May 2024', '4 May 2024']
Length: 4, dtype: str
Fecha MM.DD.AAAA (mes>12 en 2 posición): 4 filas

Rango de valores numéricos:
     Weight kg  Body Length cm   Latitude  Longitude
min     -0.252           -19.0 -78.582973  11.074008
max   1100.000           350.0  52.853843  34.896734

Peso <= 0 : 5 longitudes <=0: 6
Cordenasdas fuera de Europa Central (lat<40 o lon>30): 6


In [ ]:
df = pd.read_csv(r"c:\Users\sp9\OneDrive\Documentos\animal_data.csv")  
df.columns = ["animal_type", "country", "weight_kg", "body_length_cm", "gender",
              "animal_code", "latitude", "longitude", "animal_name", "observation_date"]
df = df.drop(columns="animal_code")

vacias = df.drop(columns="observation_date").isna().all(axis=1)
print("Filas vacías eliminadas:", vacias.sum())
df = df.loc[~vacias].reset_index(drop=True)

Filas vacías eliminadas: 12


In [40]:
tipo_map = {
    "red squirrel": "red squirrel", "red squirel": "red squirrel", "red squirrell": "red squirrel",
    "european bison": "European bison", "european bisson": "European bison", "european buster": "European bison",
    "lynx": "lynx",
    "hedgehog": "hedgehog", "ledgehod": "hedgehog", "wedgehod": "hedgehog",
    }
limpio = df["animal_type"].str.lower().str.replace(r"[^a-z ]", "", regex=True).str.strip()
assert limpio.dropna().isin(tipo_map).all(), "hay etiquetas sin mapear"
df["animal_type"] = limpio.map(tipo_map)
pais_map = {"PL": "Poland", "HU": "Hungary", "Hungry": "Hungary", "CZ": "Czech Republic",
            "CC": "Czech Republic", "Czech": "Czech Republic", "DE": "Germany", "Australia": "Austria"}
df["country"] = df["country"].str.strip().replace(pais_map)
df["gender"] = df["gender"].fillna("not determined")
df["animal_name"] = df["animal_name"].fillna("unnamed")
print(df["animal_type"].value_counts(dropna=False))
print(df["country"].value_counts(dropna=False))

animal_type
red squirrel      579
hedgehog          278
lynx               71
European bison     63
NaN                 8
Name: count, dtype: int64
country
Poland            306
Germany           181
Hungary           167
Slovakia          151
Czech Republic    116
Austria            78
Name: count, dtype: int64


In [44]:
s = df["observation_date"].str.strip()
dmy = pd.to_datetime(s, format="%d.%m.%Y", errors="coerce")
mdy = pd.to_datetime(s, format="%m.%d.%Y", errors="coerce")
txt = pd.to_datetime(s.str.title(), format="%d %b %Y", errors="coerce")

inequivoca = txt.copy()
inequivoca = inequivoca.fillna(dmy.where(mdy.isna() | (dmy == mdy)))
inequivoca = inequivoca.fillna(mdy.where(dmy.isna()))
ref = inequivoca.ffill().bfill()

usar_dmy = (dmy - ref).abs() <= (mdy - ref).abs()
usar_dmy = usar_dmy.where(mdy.notna(), True).where(dmy.notna(), False)
df["observation_date"] = txt.fillna(dmy.where(usar_dmy, mdy))
print(df["observation_date"].dtype, "|", df["observation_date"].min().date(), "->", df["observation_date"].max().date())
print("Fecha no convertida:", df["observation_date"].isna().sum())
print("Filas en orden cronológico:", f"{(df['observation_date'].diff().dt.days.fillna(0) >= 0).mean(): .1%}")


datetime64[us] | 2024-03-01 -> 2024-09-03
Fecha no convertida: 0
Filas en orden cronológico:  99.9%


In [45]:
print("Duplicados antes", df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
print("Duplicados después:", df.duplicated().sum(), "| filas restantes:", len(df))

Duplicados antes 166
Duplicados después: 0 | filas restantes: 833


In [53]:
import numpy as np

df["weight_kg"] = df["weight_kg"].abs()
df["body_length_cm"] = df["body_length_cm"].abs()

fuera = ~df["latitude"].between(45, 55) | ~df["longitude"].between(9, 25)
fuera &= df["latitude"].notna()
print("Coordenadas imposibles anuladas:", fuera.sum())
df.loc[fuera, ["latitude", "longitude"]] = np.nan

df["animal_type"] = np.where(df["animal_type"].isna() & (df["weight_kg"] < 0.4), "red squirrel", df["animal_type"])
print("animal_type sin asignar:", df["animal_type"].isna().sum())

Coordenadas imposibles anuladas: 0
animal_type sin asignar: 0


In [60]:
for col in ["weight_kg", "body_length_cm"]:
    q1 = df.groupby("animal_type")[col].transform(lambda x: x.quantile(0.25))
    q3 = df.groupby("animal_type")[col].transform(lambda x: x.quantile(0.75))
    iqr = q3 - q1
    out = (df[col] < q1 - 3 * iqr) | (df[col] > q3 + 3 *iqr)
    print(f"{col}: outliers detectados = {out.sum()}")
    print(df.loc[out, ["animal_type", col]].to_string())
    df.loc[out, col] = np.nan
    
for col in ["weight_kg", "body_length_cm"]:
    df[col] = df[col].fillna(df.groupby("animal_type")[col].transform("median"))
    
print("Dimensiones:", df.shape)
print("\nTipos:"); print(df.dtypes)
print("\nFaltantes por columna"); print(df.isna().sum())
print("\nDuplicados:", df.duplicated().sum())
print("\nanimal_type:"); print(df["animal_type"].value_counts())
print("\ncountry:"); print(df["country"].value_counts())
print("\ngender:"); print(df["country"].value_counts())
print("\nMínimos y máximos por especie:")
print(df.groupby("animal_type")[["weight_kg", "body_length_cm"]].agg(["min", "max"]))
print("\nPesos/longitudes <= 0:", (df[["weight_kg", "body_length_cm"]] <= 0).sum().sum())

df.to_csv("animal_data_clean.csv", index=False)
print("Archivo limpio guardado: animal_data_clean.csv")



weight_kg: outliers detectados = 0
Empty DataFrame
Columns: [animal_type, weight_kg]
Index: []
body_length_cm: outliers detectados = 0
Empty DataFrame
Columns: [animal_type, body_length_cm]
Index: []
Dimensiones: (833, 9)

Tipos:
animal_type                    str
country                        str
weight_kg                  float64
body_length_cm             float64
gender                         str
latitude                   float64
longitude                  float64
animal_name                    str
observation_date    datetime64[us]
dtype: object

Faltantes por columna
animal_type          0
country              0
weight_kg            0
body_length_cm       0
gender               0
latitude            85
longitude           85
animal_name          0
observation_date     0
dtype: int64

Duplicados: 0

animal_type:
animal_type
red squirrel      421
hedgehog          278
lynx               71
European bison     63
Name: count, dtype: int64

country:
country
Poland            192
Ger

In [66]:
mediana_especie = df.groupby("animal_type")["weight_kg"].transform("median")
df["peso_vs_especie"] = np.where(df["weight_kg"] > mediana_especie,"sobre la mediana", "igual o bajo la mediana")
print(df["peso_vs_especie"].value_counts())
print()
print(pd.crosstab(df["animal_type"], df["peso_vs_especie"]))

filas = []
for especie in df["animal_type"].unique():
    w = df.loc[df["animal_type"] == especie, "weight_kg"].to_numpy()
    filas.append({"especie": especie, "n": w.size,
                  "media_kg": np.nanmean(w), "mediana_kg": np.nanmedian(w),
                  "desv_std_kg": np.std(w), "coef_variacion_%": 100 * np.std(w) / np.nanmean(w)})
    resumen_np = pd.DataFrame(filas).set_index("especie").round(3)
    print(resumen_np)

obs_tipo = df.groupby("animal_type").size().sort_values(ascending=False).rename("observaciones") 
print(obs_tipo)
print("\nParticipación (%):"); print((100 * obs_tipo / obs_tipo.sum().round(1)))

prom_tipo = df.groupby("animal_type")[["weight_kg", "body_length_cm"]].mean().round(2).sort_values("weight_kg", ascending=False)
print(prom_tipo)

prom_pais = df.groupby("country").agg(registros=("weight_kg", "size"), peso_medio_kg=("weight_kg", "mean")).round(2)
print(prom_pais.sort_values("peso_medio_kg", ascending=False))

print("\nPeso medio de la ardilla roja por país:")
print(df[df["animal_type"] == "red squirrel"].groupby("country")["weight_kg"].agg(["size", "mean"]).round(3).sort_values("mean", ascending=False))

print(pd.crosstab(df["country"], df["animal_type"], margins=True))

sexo = df[df["gender"].isin(["male", "female"])].groupby(["animal_type", "gender"])["weight_kg"].mean().unstack().round(3)
sexo["dif_macho_hembra_%"] = (100 * (sexo["male"] - sexo["female"]) / sexo["female"]).round(1)
print(sexo)

por_mes = df.groupby(df["observation_date"].dt.to_period("M")) .size().rename("observaciones")
print(por_mes) 


peso_vs_especie
igual o bajo la mediana    464
sobre la mediana           369
Name: count, dtype: int64

peso_vs_especie  igual o bajo la mediana  sobre la mediana
animal_type                                               
European bison                        32                31
hedgehog                             175               103
lynx                                  40                31
red squirrel                         217               204
                 n  media_kg  mediana_kg  desv_std_kg  coef_variacion_%
especie                                                                
European bison  63   588.397       603.0      242.084            41.143
                 n  media_kg  mediana_kg  desv_std_kg  coef_variacion_%
especie                                                                
European bison  63   588.397       603.0      242.084            41.143
lynx            71    21.169        22.0        4.699            22.199
                  n  media_kg  median